<a href="https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammadayaanahsan/flyRank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

One row = one content page (content_id), measured over a fixed
90-day trailing window (the `_90d` columns: impressions_90d,
sessions_90d, etc.), as provided in the starter dataset. This is
verified below by loading the data and checking there is exactly
one row per content_id, with `_90d` columns representing a fixed
recent window rather than the page's entire history.

In [1]:
!git clone https://github.com/mohammadayaanahsan/flyRank-internship.git
%cd flyRank-internship

Cloning into 'flyRank-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 129 (delta 45), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.85 MiB | 8.10 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/flyRank-internship


In [2]:
# (Run the git clone cell first if this is a fresh session)
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Total rows: {len(df)}")
print(f"Unique content_id values: {df['content_id'].nunique()}")
print(f"One row per content page: {len(df) == df['content_id'].nunique()}")

print(df[["content_id", "impressions_90d", "sessions_90d", "days_since_last_update"]].head(5))

Total rows: 30000
Unique content_id values: 30000
One row per content page: True
             content_id  impressions_90d  sessions_90d  days_since_last_update
0  content_304f48230142             3803            17                      20
1  content_a1fb4e703a9e            15320             9                      25
2  content_9aa793d4d895            12581            11                      20
3  content_331d6c4de07b            11751            78                      22
4  content_d99b7a2d90ca            19140           145                      14


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Features** (observable signals known before the decision point):
impressions_90d, sessions_90d, avg_position, ctr, word_count,
content_age_days, days_since_last_update, engagement_rate,
scroll_rate.

**Label/proxy:** trend_direction (used to build is_declining_label =
trend_direction == "down"). This is a proxy, not a guaranteed future
outcome.

**Context (not used as a feature or label):** content_id, client_id
- these are join keys for grouping/identification only, not signal.

**Excluded:** any product decision fields (health_score,
priority_score, action_type) are excluded because they are not
shipped in this dataset, and even if they were, using them as
features would let the model just copy an existing rule instead of
learning from raw signals - this would be circular, not a discovery.

In [3]:
print("All columns in starter dataset:")
print(list(df.columns))

All columns in starter dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 3. Verify it with queries (grain, counts, missing values, windows)

Below, the grain (one row per content page) and the health of the
_90d fields are checked directly, including missing values.

In [4]:
print(f"Rows: {len(df)}, Unique content_id: {df['content_id'].nunique()}")
print()
print("Missing values in key columns:")
print(df[["impressions_90d", "sessions_90d", "avg_position", "ctr",
          "word_count", "days_since_last_update", "trend_direction"]].isnull().sum())
print()
print("trend_direction value counts:")
print(df["trend_direction"].value_counts())

Rows: 30000, Unique content_id: 30000

Missing values in key columns:
impressions_90d              0
sessions_90d                 0
avg_position                 0
ctr                          0
word_count                7699
days_since_last_update       0
trend_direction              0
dtype: int64

trend_direction value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This starter dataset is a small (~30,000-row) anonymized slice, not
the full warehouse - so results here are directional, not
production-scale evidence. The `_90d` columns are a single fixed
window per page, so there's no way to see how a page's signals
changed over time within this file alone (that requires the
warehouse's daily fact table). The trend_direction label is computed
from the current window, not a genuinely future outcome, so it can't
tell us whether a page will actually decline going forward - only
whether it currently looks like it's declining. Finally, since this
is anonymized and aggregated, it cannot tell us anything about which
specific client or real-world page is affected.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.